
This notebook implements an intelligent multi-agent system that:
- **Loads** scanned PDF documents
- **Classifies** them as `Cease`, `Uncertain`, or `Irrelevant`
- **Routes** each document to the appropriate agent
- **Supports Human-in-the-Loop (HITL)** for uncertain cases



Install Dependencies

In [1]:
# Install required packages
# langgraph-checkpoint-sqlite is the separate package needed for SqliteSaver in LangGraph >= 0.2
!pip install -q langchain langchain-google-genai langchain-community langgraph \
               langgraph-checkpoint-sqlite \
               pymupdf python-dotenv pydantic google-generativeai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


 Cell 2: Imports & Configuration

In [2]:
import os
import json
import sqlite3
import datetime
from pathlib import Path
from typing import TypedDict, Literal, Annotated, List, Optional

# LangChain — Gemini integration
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
# PyMuPDFLoader import path changed between langchain versions — handle both
try:
    from langchain_community.document_loaders import PyMuPDFLoader
except ImportError:
    from langchain.document_loaders import PyMuPDFLoader

# LangGraph
from langgraph.graph import StateGraph, END
import operator

# SqliteSaver moved to its own package in LangGraph >= 0.2
# This block handles both old and new import paths automatically
try:
    from langgraph.checkpoint.sqlite import SqliteSaver          # LangGraph < 0.2
except ImportError:
    from langgraph.checkpoint.sqlite import SqliteSaver          # langgraph-checkpoint-sqlite


# Pydantic for structured output
from pydantic import BaseModel, Field

print("✅ All imports successful!")

✅ All imports successful!


 Cell 3: Gemini API Key & LLM Setup


In [3]:
import getpass
import time
import re
import google.generativeai as genai

# ─── Set your Gemini API key ─────────────────────────────────────────
# Option A: Hard-code (dev only)
# os.environ["GOOGLE_API_KEY"] = "AIza..."
# Option B: .env file
# from dotenv import load_dotenv; load_dotenv()
# Option C: prompt at runtime (used here)
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass(
        "Enter your Google Gemini API key (free at aistudio.google.com): "
    )
API_KEY = os.environ["GOOGLE_API_KEY"]
genai.configure(api_key=API_KEY)

# ─── Step 1: List every model your key supports ──────────────────────
print("🔍 Models available on your API key:\n")
available = []
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        available.append(m.name)
        print(f"   {m.name}")

if not available:
    raise RuntimeError("No models found — check your API key is valid.")

# ─── Step 2: Probe each candidate to find one with REAL free quota ───
# Order: prefer flash-lite (lowest quota usage) → flash → pro
# 'limit: 0' in the 429 error means that specific model has NO free quota.
PROBE_ORDER = [
    "models/gemini-2.0-flash-lite",     # cheapest quota cost
    "models/gemini-2.0-flash-lite-001",
    "models/gemini-1.5-flash-8b",       # smallest 1.5 variant
    "models/gemini-1.5-flash-8b-001",
    "models/gemini-1.5-flash",
    "models/gemini-1.5-flash-latest",
    "models/gemini-1.5-flash-001",
    "models/gemini-2.0-flash",
    "models/gemini-2.0-flash-001",
    "models/gemini-1.5-pro",
    "models/gemini-1.5-pro-latest",
]

# Only probe models that are actually in the available list
candidates = [m for m in PROBE_ORDER if m in available]
# Append any remaining available models not in our list
candidates += [m for m in available if m not in candidates]

print("\n🔬 Probing models for available free quota...")
llm = None
model_id = None
for candidate in candidates:
    mid = candidate.replace("models/", "")
    try:
        _test_llm = ChatGoogleGenerativeAI(
            model=mid,
            temperature=0,
            google_api_key=API_KEY,
        )
        _test_llm.invoke("hi")      # real call — triggers 429 if quota is zero
        llm = _test_llm
        model_id = mid
        print(f"   ✅ {mid} — quota OK")
        break
    except Exception as e:
        err = str(e)
        if "429" in err or "RESOURCE_EXHAUSTED" in err:
            # Try to read the retry-after hint from the error message
            wait_match = re.search(r"retry in ([\d.]+)s", err, re.IGNORECASE)
            hint = f" (retry in {float(wait_match.group(1)):.0f}s)" if wait_match else ""
            print(f"   ⏳ {mid} — quota exhausted{hint}, skipping")
        elif "404" in err or "NOT_FOUND" in err:
            print(f"   ❌ {mid} — not found on this key, skipping")
        else:
            print(f"   ⚠️  {mid} — error: {err[:120]}")

if llm is None:
    print("""
╔══════════════════════════════════════════════════════════════╗
║  ALL MODELS QUOTA EXHAUSTED — common causes & fixes:        ║
║                                                              ║
║  1. Daily free limit hit → wait until midnight Pacific time  ║
║  2. Wrong project → create a NEW project at                  ║
║     console.cloud.google.com and generate a fresh API key    ║
║  3. Billing not enabled → add a card at                      ║
║     console.cloud.google.com/billing (free $300 credit)      ║
║  4. Check your usage → https://ai.dev/rate-limit             ║
╚══════════════════════════════════════════════════════════════╝
""")
    raise RuntimeError("No Gemini model with available quota found. See suggestions above.")

print(f"\n🎯 Using model: {model_id}")

# ─── Step 3: Rate-limit aware invoke helper ──────────────────────────
# Cannot monkey-patch llm.invoke (Pydantic model forbids it).
# Instead define llm_invoke() — all agent nodes call this instead of llm.invoke().
def llm_invoke(prompt, **kwargs):
    """Call llm.invoke with automatic retry on 429 / RESOURCE_EXHAUSTED."""
    for attempt in range(3):
        try:
            return llm.invoke(prompt, **kwargs)
        except Exception as e:
            if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < 2:
                wait_match = re.search(r"retry in ([\d.]+)s", str(e), re.IGNORECASE)
                wait = float(wait_match.group(1)) + 2 if wait_match else 60
                print(f"   ⏳ Rate limited — waiting {wait:.0f}s then retrying (attempt {attempt+2}/3)...")
                time.sleep(wait)
            else:
                raise

print("✅ llm_invoke() ready — auto-retries on 429 rate limit errors")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Enter your Google Gemini API key (free at aistudio.google.com): ··········
🔍 Models available on your API key:

   models/gemini-2.5-flash
   models/gemini-2.5-pro
   models/gemini-2.0-flash
   models/gemini-2.0-flash-001
   models/gemini-2.0-flash-lite-001
   models/gemini-2.0-flash-lite
   models/gemini-2.5-flash-preview-tts
   models/gemini-2.5-pro-preview-tts
   models/gemma-3-1b-it
   models/gemma-3-4b-it
   models/gemma-3-12b-it
   models/gemma-3-27b-it
   models/gemma-3n-e4b-it
   models/gemma-3n-e2b-it
   models/gemini-flash-latest
   models/gemini-flash-lite-latest
   models/gemini-pro-latest
   models/gemini-2.5-flash-lite
   models/gemini-2.5-flash-image
   models/gemini-2.5-flash-lite-preview-09-2025
   models/gemini-3-pro-preview
   models/gemini-3-flash-preview
   models/gemini-3.1-pro-preview
   models/gemini-3.1-pro-preview-customtools
   models/gemini-3.1-flash-lite-preview
   models/gemini-3-pro-image-preview
   models/nano-banana-pro-preview
   models/gemini-3.1-flas

##State Schema (LangGraph)

In [4]:
class DocumentState(TypedDict):
    """Shared state passed between all nodes in the graph."""
    # Input
    pdf_path: str                              # Path to the PDF file
    document_name: str                         # File name for logging
    date_received: str                         # ISO date string

    # Extracted
    raw_text: str                              # Full text extracted from PDF

    # Classification
    classification: Optional[str]              # "cease" | "uncertain" | "irrelevant"
    classification_reason: Optional[str]       # Explanation from classifier
    confidence: Optional[float]                # 0.0–1.0

    # Extracted details (for cease docs)
    extracted_details: Optional[dict]

    # HITL
    hitl_required: bool                        # True if uncertain
    human_decision: Optional[str]              # Human override
    human_notes: Optional[str]

    # Audit trail (accumulates across all nodes)
    audit_log: Annotated[List[str], operator.add]

    # Final status
    status: Optional[str]                      # "completed" | "error"

print("✅ DocumentState schema defined")

✅ DocumentState schema defined


##  Pydantic Schemas for Structured Output

In [5]:
class ClassificationResult(BaseModel):
    """Structured result from the classification agent."""
    classification: Literal["cease", "uncertain", "irrelevant"] = Field(
        description="Document classification: cease, uncertain, or irrelevant"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence score between 0.0 and 1.0"
    )
    reason: str = Field(
        description="Short explanation of why this classification was chosen"
    )
    key_phrases: List[str] = Field(
        description="Key phrases from the document that support this classification"
    )


class CeaseDetails(BaseModel):
    """Structured details extracted from a valid cease & desist document."""
    client_name: Optional[str] = Field(default=None, description="Primary client name")
    co_client_name: Optional[str] = Field(default=None, description="Co-client name if present")
    law_firm: Optional[str] = Field(default=None, description="Representing law firm or attorney")
    document_date: Optional[str] = Field(default=None, description="Date stated on the document")
    cease_scope: str = Field(description="What communications must stop")
    contact_redirect: Optional[str] = Field(default=None, description="Who future contact should be directed to")
    account_reference: Optional[str] = Field(default=None, description="Account or reference number")
    effective_until: Optional[str] = Field(default=None, description="Expiry of the cease, if stated")

print("✅ Pydantic schemas defined")

✅ Pydantic schemas defined


## Database & Archive Setup

In [6]:
# ── File paths ────────────────────────────────────────────────────────
DB_PATH       = "cease_desist.db"
ARCHIVE_FILE  = "irrelevant_docs.txt"
AUDIT_FILE    = "audit_log.jsonl"
CHECKPOINT_DB = "checkpoints.db"

def init_database():
    """Create cease_requests table if it doesn't already exist."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS cease_requests (
            id                   INTEGER PRIMARY KEY AUTOINCREMENT,
            date_received        TEXT NOT NULL,
            document_name        TEXT NOT NULL,
            client_name          TEXT,
            co_client_name       TEXT,
            law_firm             TEXT,
            document_date        TEXT,
            cease_scope          TEXT,
            contact_redirect     TEXT,
            account_reference    TEXT,
            classification_reason TEXT,
            human_reviewed       INTEGER DEFAULT 0,
            created_at           TEXT DEFAULT (datetime('now'))
        )
    """)
    conn.commit()
    conn.close()
    print(f"✅ SQLite database ready  : {DB_PATH}")
    print(f"✅ Archive file           : {ARCHIVE_FILE}")
    print(f"✅ Audit log              : {AUDIT_FILE}")

init_database()

✅ SQLite database ready  : cease_desist.db
✅ Archive file           : irrelevant_docs.txt
✅ Audit log              : audit_log.jsonl


## Agent Nodes

All nodes share the single `llm` instance initialised in Cell 3 (auto-selected Gemini model).


In [7]:
# ══════════════════════════════════════════════════════════════════════
#  NODE 1 — Document Loader Agent
# ══════════════════════════════════════════════════════════════════════
def document_loader_node(state: DocumentState) -> dict:
    """Load and extract text from the PDF using PyMuPDF."""
    pdf_path = state["pdf_path"]
    doc_name = state.get("document_name", Path(pdf_path).name)

    try:
        loader = PyMuPDFLoader(pdf_path)
        pages  = loader.load()
        raw_text = "\n".join(p.page_content for p in pages).strip()
        log = f"[LOADER] '{doc_name}' loaded — {len(pages)} page(s), {len(raw_text)} chars"
        print(log)
        return {"raw_text": raw_text, "document_name": doc_name, "audit_log": [log]}
    except Exception as e:
        log = f"[LOADER] ERROR loading '{doc_name}': {e}"
        print(log)
        return {"raw_text": "", "audit_log": [log], "status": "error"}


# ══════════════════════════════════════════════════════════════════════
#  NODE 2 — Classification Agent  (Gemini)
# ══════════════════════════════════════════════════════════════════════
classification_llm = llm.with_structured_output(ClassificationResult)

def invoke_classification(messages):
    """Call classification_llm with 429 retry logic."""
    for attempt in range(3):
        try:
            return classification_llm.invoke(messages)
        except Exception as e:
            if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < 2:
                wait_match = re.search(r"retry in ([\d.]+)s", str(e), re.IGNORECASE)
                wait = float(wait_match.group(1)) + 2 if wait_match else 60
                print(f"   ⏳ Rate limited — waiting {wait:.0f}s (attempt {attempt+2}/3)...")
                time.sleep(wait)
            else:
                raise

# NOTE: system + human are merged into one human message by the Gemini adapter
CLASSIFICATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert legal document classifier specialising in Cease & Desist requests.

Classify the document into EXACTLY one of three categories:

• cease      — The document CLEARLY and EXPLICITLY requests cessation of all
               communications. Look for: "cease and desist", "stop all contact",
               "do not contact", directives redirecting communication to a law firm.

• uncertain  — The document hints at stopping contact but is AMBIGUOUS, conditional,
               or only requests clarification rather than making a firm demand.

• irrelevant — The document has NOTHING to do with ceasing communications
               (e.g. Power of Attorney notice, informational letter, billing doc).

Return structured output: classification, confidence (0.0–1.0), reason, key_phrases.
"""),
    ("human", "Classify this document:\n\n{text}")
])

def classification_node(state: DocumentState) -> dict:
    """Classify the document using Gemini."""
    raw_text = state.get("raw_text", "")
    doc_name = state["document_name"]

    if not raw_text:
        log = f"[CLASSIFIER] No text to classify for '{doc_name}' — marking uncertain"
        return {"classification": "uncertain", "confidence": 0.0,
                "classification_reason": "No text extracted from document",
                "hitl_required": True, "audit_log": [log]}

    result: ClassificationResult = invoke_classification(
        CLASSIFICATION_PROMPT.format_messages(text=raw_text[:6000])
    )

    # Force HITL for uncertain label OR low confidence
    hitl_required = (result.classification == "uncertain") or (result.confidence < 0.75)

    log = (
        f"[CLASSIFIER] '{doc_name}' → {result.classification.upper()} "
        f"(confidence={result.confidence:.0%}) | {result.reason}"
    )
    print(log)

    return {
        "classification":        result.classification,
        "classification_reason": result.reason,
        "confidence":            result.confidence,
        "hitl_required":         hitl_required,
        "audit_log":             [log],
    }


# ══════════════════════════════════════════════════════════════════════
#  NODE 3 — Detail Extraction Agent  (Gemini)
# ══════════════════════════════════════════════════════════════════════
extraction_llm = llm.with_structured_output(CeaseDetails)

def invoke_extraction(messages):
    """Call extraction_llm with 429 retry logic."""
    for attempt in range(3):
        try:
            return extraction_llm.invoke(messages)
        except Exception as e:
            if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < 2:
                wait_match = re.search(r"retry in ([\d.]+)s", str(e), re.IGNORECASE)
                wait = float(wait_match.group(1)) + 2 if wait_match else 60
                print(f"   ⏳ Rate limited — waiting {wait:.0f}s (attempt {attempt+2}/3)...")
                time.sleep(wait)
            else:
                raise

EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
You are a legal data extraction specialist. Extract all structured details from
cease & desist documents. Use null for any field not explicitly stated in the text.
"""),
    ("human", "Extract structured details from this cease & desist document:\n\n{text}")
])

def extraction_node(state: DocumentState) -> dict:
    """Extract structured details from a cease document using Gemini."""
    raw_text = state.get("raw_text", "")
    doc_name = state["document_name"]

    details: CeaseDetails = invoke_extraction(
        EXTRACTION_PROMPT.format_messages(text=raw_text[:6000])
    )

    log = f"[EXTRACTOR] Details extracted from '{doc_name}' — client: {details.client_name}"
    print(log)

    return {"extracted_details": details.dict(), "audit_log": [log]}


# ══════════════════════════════════════════════════════════════════════
#  NODE 4 — Database Agent
# ══════════════════════════════════════════════════════════════════════
def database_agent_node(state: DocumentState) -> dict:
    """Insert a validated cease request into the SQLite database."""
    details        = state.get("extracted_details") or {}
    human_reviewed = 1 if state.get("human_decision") else 0

    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        INSERT INTO cease_requests
          (date_received, document_name, client_name, co_client_name,
           law_firm, document_date, cease_scope, contact_redirect,
           account_reference, classification_reason, human_reviewed)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)
    """, (
        state.get("date_received"),
        state["document_name"],
        details.get("client_name"),
        details.get("co_client_name"),
        details.get("law_firm"),
        details.get("document_date"),
        details.get("cease_scope"),
        details.get("contact_redirect"),
        details.get("account_reference"),
        state.get("classification_reason"),
        human_reviewed,
    ))
    conn.commit()
    conn.close()

    log = f"[DATABASE] Stored cease request for '{state['document_name']}'"
    print(log)
    return {"status": "completed", "audit_log": [log]}


# ══════════════════════════════════════════════════════════════════════
#  NODE 5 — Archiving Agent
# ══════════════════════════════════════════════════════════════════════
def archiving_agent_node(state: DocumentState) -> dict:
    """Append irrelevant document metadata to the flat archive file."""
    line = (
        f"{state.get('date_received')} | "
        f"{state['document_name']} | "
        f"Reason: {state.get('classification_reason', 'N/A')}\n"
    )
    with open(ARCHIVE_FILE, "a") as f:
        f.write(line)

    log = f"[ARCHIVE] Archived irrelevant doc '{state['document_name']}'"
    print(log)
    return {"status": "completed", "audit_log": [log]}


# ══════════════════════════════════════════════════════════════════════
#  NODE 6 — HITL Node  (pause for human review)
# ══════════════════════════════════════════════════════════════════════
def hitl_node(state: DocumentState) -> dict:
    """Present the uncertain document to a human reviewer and collect their decision."""
    doc_name       = state["document_name"]
    classification = state.get("classification") or "unknown"
    confidence     = state.get("confidence")   # may be None — handled below
    reason         = state.get("classification_reason") or "No reason provided"
    conf_str       = f"{confidence:.0%}" if confidence is not None else "N/A"

    print("\n" + "═" * 62)
    print("🔍  HUMAN REVIEW REQUIRED")
    print("═" * 62)
    print(f"  Document  : {doc_name}")
    print(f"  AI Label  : {classification.upper()} (confidence={conf_str})")
    print(f"  AI Reason : {reason}")
    print("─" * 62)
    print("  Document Preview:")
    print("  " + state.get("raw_text", "")[:600].replace("\n", "\n  "))
    print("═" * 62)

    while True:
        decision = input("\n  Your decision [cease / irrelevant / skip]: ").strip().lower()
        if decision in ("cease", "irrelevant", "skip"):
            break
        print("  ⚠️  Please enter 'cease', 'irrelevant', or 'skip'")

    notes = input("  Optional notes (Enter to skip): ").strip()

    if decision == "skip":
        decision = classification   # keep the AI's label

    log = f"[HITL] Human reviewed '{doc_name}': decision={decision.upper()}, notes='{notes}'"
    print(log)

    return {
        "human_decision": decision,
        "human_notes":    notes,
        "classification": decision,   # override
        "audit_log":      [log],
    }


# ══════════════════════════════════════════════════════════════════════
#  NODE 7 — Audit Agent
# ══════════════════════════════════════════════════════════════════════
def audit_agent_node(state: DocumentState) -> dict:
    """Write a complete audit record to the JSONL audit log."""
    record = {
        "timestamp":      datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "document_name":  state["document_name"],
        "date_received":  state.get("date_received"),
        "classification": state.get("classification"),
        "confidence":     state.get("confidence"),
        "reason":         state.get("classification_reason"),
        "human_reviewed": bool(state.get("human_decision")),
        "human_decision": state.get("human_decision"),
        "human_notes":    state.get("human_notes"),
        "status":         state.get("status"),
        "audit_trail":    state.get("audit_log", []),
    }
    with open(AUDIT_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")

    log = f"[AUDIT] Record written for '{state['document_name']}'"
    print(log)
    return {"audit_log": [log]}


print("✅ All 7 agent nodes defined")

✅ All 7 agent nodes defined


Cell 8: Routing Logic

In [8]:
def route_after_classification(state: DocumentState) -> str:
    """
    After classification decide where to route:
    • uncertain OR low confidence  → hitl
    • cease (high confidence)      → extract_details
    • irrelevant (high confidence) → archive
    """
    if state.get("hitl_required"):
        return "hitl"
    if state.get("classification") == "cease":
        return "extract_details"
    return "archive"


def route_after_hitl(state: DocumentState) -> str:
    """
    After human review:
    • cease      → extract_details
    • irrelevant → archive
    """
    decision = state.get("human_decision", state.get("classification"))
    return "extract_details" if decision == "cease" else "archive"


print("✅ Routing functions defined")

✅ Routing functions defined


 Build the LangGraph

In [9]:
# ── Checkpointer (SQLite-backed persistence) ──────────────────────────
# LangGraph >= 0.2 requires the separate langgraph-checkpoint-sqlite package
# and uses a context-manager style. This block handles both versions cleanly.
try:
    # New style: langgraph-checkpoint-sqlite package (LangGraph >= 0.2)
    from langgraph.checkpoint.sqlite import SqliteSaver
    import sqlite3 as _sqlite3
    ckpt_conn    = _sqlite3.connect(CHECKPOINT_DB, check_same_thread=False)
    checkpointer = SqliteSaver(ckpt_conn)
    print("✅ Checkpointer: langgraph-checkpoint-sqlite (new path)")
except Exception as e:
    # Fallback: in-memory checkpointer (no persistence, but graph still works)
    from langgraph.checkpoint.memory import MemorySaver
    checkpointer = MemorySaver()
    print(f"⚠️  SqliteSaver unavailable ({e}). Using MemorySaver (no disk persistence).")

# ── Build graph ───────────────────────────────────────────────────────
builder = StateGraph(DocumentState)

# Register nodes
builder.add_node("load_document",   document_loader_node)
builder.add_node("classify",        classification_node)
builder.add_node("hitl",            hitl_node)
builder.add_node("extract_details", extraction_node)
builder.add_node("store_db",        database_agent_node)
builder.add_node("archive",         archiving_agent_node)
builder.add_node("audit",           audit_agent_node)

# Entry point
builder.set_entry_point("load_document")

# Deterministic edges
builder.add_edge("load_document",   "classify")
builder.add_edge("extract_details", "store_db")
builder.add_edge("store_db",        "audit")
builder.add_edge("archive",         "audit")
builder.add_edge("audit",           END)

# Conditional edges (routing)
builder.add_conditional_edges(
    "classify",
    route_after_classification,
    {"hitl": "hitl", "extract_details": "extract_details", "archive": "archive"}
)
builder.add_conditional_edges(
    "hitl",
    route_after_hitl,
    {"extract_details": "extract_details", "archive": "archive"}
)

# Compile with checkpointer
graph = builder.compile(checkpointer=checkpointer)

print("✅ LangGraph compiled successfully")
print("   Nodes:", list(graph.nodes.keys()))

✅ Checkpointer: langgraph-checkpoint-sqlite (new path)
✅ LangGraph compiled successfully
   Nodes: ['__start__', 'load_document', 'classify', 'hitl', 'extract_details', 'store_db', 'archive', 'audit']


## 📊 Cell 10: Visualize the Graph

In [10]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: print Mermaid source for copy-paste into mermaid.live
    print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	load_document(load_document)
	classify(classify)
	hitl(hitl)
	extract_details(extract_details)
	store_db(store_db)
	archive(archive)
	audit(audit)
	__end__([<p>__end__</p>]):::last
	__start__ --> load_document;
	archive --> audit;
	classify -.-> archive;
	classify -.-> extract_details;
	classify -.-> hitl;
	extract_details --> store_db;
	hitl -.-> archive;
	hitl -.-> extract_details;
	load_document --> classify;
	store_db --> audit;
	audit --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## ▶️ Cell 11: Main Processing Function

In [11]:
def process_document(pdf_path: str, thread_id: Optional[str] = None) -> dict:
    """
    Run the full Gemini-powered pipeline for a single PDF document.

    Args:
        pdf_path  : Local path to the PDF file.
        thread_id : Optional LangGraph thread ID for checkpointing/resuming.

    Returns:
        Final DocumentState dict.
    """
    doc_name  = Path(pdf_path).name
    thread_id = thread_id or f"{doc_name}-{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M%S')}"

    initial_state: DocumentState = {
        "pdf_path":           pdf_path,
        "document_name":      doc_name,
        "date_received":      datetime.date.today().isoformat(),
        "raw_text":           "",
        "classification":     None,
        "classification_reason": None,
        "confidence":         None,
        "extracted_details":  None,
        "hitl_required":      False,
        "human_decision":     None,
        "human_notes":        None,
        "audit_log":          [],
        "status":             None,
    }

    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n{'═'*62}")
    print(f"  📄 Processing: {doc_name}")
    print(f"  🤖 Model     : Gemini 1.5 Flash (free tier)")
    print(f"{'═'*62}")

    final_state = graph.invoke(initial_state, config)

    label = str(final_state.get('classification', '?')).upper()
    print(f"\n  ✅ Done — {doc_name} → {label}")
    return final_state


print("✅ process_document() ready")

✅ process_document() ready


## 🧪 Cell 12: Process Sample Documents

Based on the 3 uploaded sample PDFs:

| File | Expected Result |
|---|---|
| `sample3.pdf` | Five Lakes Law Group — **CEASE** |
| `sample2.pdf` | Harridan & Sutherland — **UNCERTAIN** → HITL |
| `sample1.pdf` | Abernathy & Rowe POA — **IRRELEVANT** |

In [12]:
import sys

# ── Detect environment: Google Colab vs Local ─────────────────────────
IN_COLAB = "google.colab" in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"🖥️  Environment: {'Google Colab' if IN_COLAB else 'Local (Jupyter)'}")

# ── COLAB: upload PDFs directly from your computer ────────────────────
if IN_COLAB:
    from google.colab import files
    print("\n📤 Please upload your PDF files using the dialog below.")
    print("   Select sample1.pdf, sample2.pdf, sample3.pdf together.\n")
    uploaded = files.upload()   # opens file picker in Colab

    # Files land in /content/ — build the list from what was uploaded
    SAMPLE_PDFS = [f"/content/{fname}" for fname in uploaded.keys() if fname.endswith(".pdf")]

    if not SAMPLE_PDFS:
        print("⚠️  No PDF files uploaded. Upload at least one .pdf and re-run this cell.")
    else:
        print(f"✅ {len(SAMPLE_PDFS)} PDF(s) uploaded to /content/:")
        for p in SAMPLE_PDFS:
            print(f"   {p}")

# ── LOCAL (Windows / Mac / Linux): set your folder path ──────────────
else:
    # ⚠️  WINDOWS USERS: always use Path() — never type backslashes directly!
    # Backslash paths like "C:\Users\..." cause SyntaxError.
    # Path() handles Windows separators automatically.

    # Option A: PDFs are in the SAME folder as this notebook
    BASE_DIR = Path(".")

    # Option B: PDFs are in a specific folder — edit this path:
    # BASE_DIR = Path("C:/Users/LENOVO/Downloads/agentic-ai-training-main/day5/capstone-project")

    SAMPLE_PDFS = [
        str(BASE_DIR / "sample1.pdf"),
        str(BASE_DIR / "sample2.pdf"),
        str(BASE_DIR / "sample3.pdf"),
    ]

    print(f"📁 Looking for PDFs in: {BASE_DIR.resolve()}")
    for p in SAMPLE_PDFS:
        status = "✅ found" if Path(p).exists() else "❌ NOT found"
        print(f"   {status}: {Path(p).name}")

# ── Process all found PDFs ────────────────────────────────────────────
results = []
for pdf in SAMPLE_PDFS:
    if Path(pdf).exists():
        result = process_document(pdf)
        results.append(result)
    else:
        print(f"⚠️  Skipping (not found): {pdf}")

print(f"\n✅ Processed {len(results)} document(s)")

🖥️  Environment: Google Colab

📤 Please upload your PDF files using the dialog below.
   Select sample1.pdf, sample2.pdf, sample3.pdf together.



Saving 01_copyright_infringement_photography.pdf to 01_copyright_infringement_photography.pdf
Saving 02_trademark_infringement_tech.pdf to 02_trademark_infringement_tech.pdf
Saving 03_trade_secret_misappropriation.pdf to 03_trade_secret_misappropriation.pdf
Saving 04_defamation_online_review.pdf to 04_defamation_online_review.pdf
Saving 05_patent_infringement_medical_device.pdf to 05_patent_infringement_medical_device.pdf
Saving 06_harassment_workplace.pdf to 06_harassment_workplace.pdf
Saving 07_software_license_violation.pdf to 07_software_license_violation.pdf
Saving 08_non_compete_violation.pdf to 08_non_compete_violation.pdf
Saving 09_copyright_infringement_music.pdf to 09_copyright_infringement_music.pdf
Saving 10_breach_of_contract_nda.pdf to 10_breach_of_contract_nda.pdf
Saving bw_doc_1.pdf to bw_doc_1.pdf
Saving bw_doc_2.pdf to bw_doc_2.pdf
Saving bw_doc_3.pdf to bw_doc_3.pdf
Saving bw_doc_4.pdf to bw_doc_4.pdf
Saving bw_doc_5.pdf to bw_doc_5.pdf
Saving LoA1.pdf to LoA1.pdf
Sa

/tmp/ipykernel_1486/3122222041.py:135: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return {"extracted_details": details.dict(), "audit_log": [log]}


[CLASSIFIER] '02_trademark_infringement_tech.pdf' → CEASE (confidence=100%) | The document is explicitly titled 'CEASE AND DESIST LETTER' and contains multiple direct demands to 'cease and desist all unlawful activity' and 'cease all use' of the infringing mark. It also directs communication to the law firm.
[EXTRACTOR] Details extracted from '02_trademark_infringement_tech.pdf' — client: Stellarion Systems Corp.
[DATABASE] Stored cease request for '02_trademark_infringement_tech.pdf'
[AUDIT] Record written for '02_trademark_infringement_tech.pdf'

  ✅ Done — 02_trademark_infringement_tech.pdf → CEASE

══════════════════════════════════════════════════════════════
  📄 Processing: 03_trade_secret_misappropriation.pdf
  🤖 Model     : Gemini 1.5 Flash (free tier)
══════════════════════════════════════════════════════════════
[LOADER] '03_trade_secret_misappropriation.pdf' loaded — 2 page(s), 3526 chars
[CLASSIFIER] '03_trade_secret_misappropriation.pdf' → CEASE (confidence=100%) | The doc

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 22.491678855s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

## 📋 Cell 13: View Database Contents

In [13]:
import pandas as pd

conn = sqlite3.connect(DB_PATH)
df   = pd.read_sql_query("SELECT * FROM cease_requests ORDER BY id DESC", conn)
conn.close()

print(f"📦  Cease requests in database: {len(df)}")
if not df.empty:
    pd.set_option("display.max_colwidth", 50)
    display(df)

📦  Cease requests in database: 10


,id,date_received,document_name,client_name,co_client_name,law_firm,document_date,cease_scope,contact_redirect,account_reference,classification_reason,human_reviewed,created_at
0,10,2026-03-23,10_breach_of_contract_nda.pdf,Helios Biotech Corp.,None,Whitfield & Chang LLP,"February 12, 2026",Immediately cease any further disclosure of He...,"Karen Whitfield, Esq.",None,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:36:04
1,9,2026-03-23,09_copyright_infringement_music.pdf,Carlos Rivera,None,None,"March 7, 2026",all unlawful activity relating to three origin...,Carlos Rivera,None,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:35:58
2,8,2026-03-23,08_non_compete_violation.pdf,LakePoint Advisors LLC,None,Hartley Weldon LLP,"February 20, 2026",all unlawful activity relating to the Non-Comp...,"Patricia Weldon, Esq.",None,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:35:51
3,7,2026-03-23,07_software_license_violation.pdf,Forge Software Inc.,None,Forge Software Inc.,"January 28, 2026",Immediately cease and desist all unlawful acti...,"Michael Brennan, Esq.",FSI-2024-03892,"The document is explicitly titled ""CEASE AND D...",0,2026-03-23 08:35:40
4,6,2026-03-23,06_harassment_workplace.pdf,Linda Fujimoto,None,None,"March 12, 2026",Immediately cease all unlawful activity relati...,Linda Fujimoto,None,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:34:43
5,5,2026-03-23,05_patent_infringement_medical_device.pdf,VitaWave Medical Technologies LLC,None,"Whitfield, Tannenbaum & Associates","February 5, 2026",immediately cease and desist all unlawful acti...,"Robert Tannenbaum, Esq.","U.S. Patent Nos. 11,234,567 and 11,345,678",The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:34:35
6,4,2026-03-23,04_defamation_online_review.pdf,Dr. Samantha Ellis,Ellis Integrative Health Clinic,None,"March 3, 2026",Immediately cease and desist all unlawful acti...,Dr. Samantha Ellis,None,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:34:28
7,3,2026-03-23,03_trade_secret_misappropriation.pdf,Quantivia Analytics Inc.,None,Crawford & Bates LLP,"January 15, 2026","Immediately cease all use, reproduction, and d...","Angela Morrison, Esq.",None,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:34:21
8,2,2026-03-23,02_trademark_infringement_tech.pdf,Stellarion Systems Corp.,None,Marks & Sullivan LLP,"February 28, 2026",Cease all use of the name 'NovaCORE Suite' or ...,"Jonathan Marks, Esq.",None,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:34:15
9,1,2026-03-23,01_copyright_infringement_photography.pdf,Priya Nair Photography LLC,None,None,"March 10, 2026",Immediately cease and desist all unlawful acti...,Priya Nair,VA-2025-004821,The document is explicitly titled 'CEASE AND D...,0,2026-03-23 08:34:08


## 📜 Cell 14: View Audit Log

In [14]:
if Path(AUDIT_FILE).exists():
    records = []
    with open(AUDIT_FILE) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    print(f"📋  Audit records: {len(records)}\n")
    for r in records:
        print(
            f"  {r['timestamp'][:19]}  |  {r['document_name']:<32}  |  "
            f"{str(r['classification']).upper():<12}  |  "
            f"human={'yes' if r['human_reviewed'] else 'no':<3}  |  "
            f"status={r['status']}"
        )
else:
    print("No audit log yet — run Cell 12 first.")

📋  Audit records: 15

  2026-03-23T08:34:08  |  01_copyright_infringement_photography.pdf  |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:34:15  |  02_trademark_infringement_tech.pdf  |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:34:21  |  03_trade_secret_misappropriation.pdf  |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:34:28  |  04_defamation_online_review.pdf   |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:34:35  |  05_patent_infringement_medical_device.pdf  |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:34:43  |  06_harassment_workplace.pdf       |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:35:41  |  07_software_license_violation.pdf  |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:35:51  |  08_non_compete_violation.pdf      |  CEASE         |  human=no   |  status=completed
  2026-03-23T08:35:58  |  09_copyright_infringement_music.pdf  | 

## 🗃️ Cell 15: View Archive File

In [15]:
if Path(ARCHIVE_FILE).exists():
    with open(ARCHIVE_FILE) as f:
        content = f.read()
    print("🗄️  Archived (irrelevant) documents:\n")
    print(content or "  (empty)")
else:
    print("No archive file yet — run Cell 12 first.")

🗄️  Archived (irrelevant) documents:

2026-03-23 | bw_doc_1.pdf | Reason: No text extracted from document
2026-03-23 | bw_doc_2.pdf | Reason: No text extracted from document
2026-03-23 | bw_doc_3.pdf | Reason: No text extracted from document
2026-03-23 | bw_doc_4.pdf | Reason: No text extracted from document
2026-03-23 | bw_doc_5.pdf | Reason: No text extracted from document



 Processing Summary

In [16]:
if results:
    print("\n" + "═" * 68)
    print("  PROCESSING SUMMARY  (Google Gemini — free tier)")
    print("═" * 68)
    print(f"  {'Document':<32} {'Classification':<14} {'Confidence':<12} {'Status'}")
    print("  " + "-" * 66)
    for r in results:
        conf  = r.get('confidence')
        conf_str = f"{conf:.0%}" if conf is not None else "N/A"
        print(
            f"  {r['document_name']:<32} "
            f"{str(r.get('classification','?')).upper():<14} "
            f"{conf_str:<12} "
            f"{r.get('status','?')}"
        )
    print("═" * 68)

    cease_count      = sum(1 for r in results if r.get("classification") == "cease")
    irrelevant_count = sum(1 for r in results if r.get("classification") == "irrelevant")
    uncertain_count  = sum(1 for r in results if r.get("classification") == "uncertain")
    human_count      = sum(1 for r in results if r.get("human_decision"))

    print(f"\n  ✅ Cease requests stored   : {cease_count}")
    print(f"  📁 Irrelevant docs archived: {irrelevant_count}")
    print(f"  ⚠️  Uncertain (sent to HITL): {uncertain_count}")
    print(f"  👤 Human reviews performed : {human_count}")


════════════════════════════════════════════════════════════════════
  PROCESSING SUMMARY  (Google Gemini — free tier)
════════════════════════════════════════════════════════════════════
  Document                         Classification Confidence   Status
  ------------------------------------------------------------------
  01_copyright_infringement_photography.pdf CEASE          100%         completed
  02_trademark_infringement_tech.pdf CEASE          100%         completed
  03_trade_secret_misappropriation.pdf CEASE          100%         completed
  04_defamation_online_review.pdf  CEASE          100%         completed
  05_patent_infringement_medical_device.pdf CEASE          100%         completed
  06_harassment_workplace.pdf      CEASE          100%         completed
  07_software_license_violation.pdf CEASE          95%          completed
  08_non_compete_violation.pdf     CEASE          100%         completed
  09_copyright_infringement_music.pdf CEASE          100%      